# Ollama Manager Usage Examples

This notebook demonstrates how to use the `OllamaDB` manager class to create, manage, and interact with an Ollama runtime in a Docker container.

## Setup

### Import Dependencies

In [ ]:
import sys
import uuid
import tempfile
from pathlib import Path
import requests

sys.path.insert(0, str(Path().cwd().parent))

from docker_db.dbs.ollama_db import OllamaConfig, OllamaDB

## Creating an Ollama Instance

Let's create a temporary directory for Ollama data and define a container configuration.
The volume is created in the OS temp directory so no container data lands inside the repo.

In [ ]:
temp_dir = Path(tempfile.mkdtemp())

container_name = f"demo-ollama-{uuid.uuid4().hex[:8]}"

config = OllamaConfig(
    project_name="demo",
    container_name=container_name,
    workdir=temp_dir,
    retries=30,
    delay=2,
)

db_manager = OllamaDB(config)

## Start the Runtime

We'll now create and start the Ollama runtime.

In [ ]:
db_manager.create_db()
print(f"Ollama started successfully in container '{container_name}'")
print("Connection details:")
print(f"  Host: {config.host}")
print(f"  Port: {config.port}")
print(f"  Base URL: {db_manager.connection_string()}")

## Query the Ollama API

Now let's call the API and inspect the available local models.

In [ ]:
response = requests.get(f"{db_manager.base_url}/api/tags", timeout=10)
response.raise_for_status()
payload = response.json()

models = payload.get("models", [])
print(f"Available local models: {len(models)}")
for model in models:
    print(" -", model.get("name", "unknown"))

## Clean Up

When you're done with the runtime, remove the container.

In [ ]:
db_manager.delete_db(running_ok=True)
print(f"Ollama container '{container_name}' deleted")

## Conclusion

This notebook demonstrated how to:

1. Configure and create an Ollama runtime with `OllamaDB`
2. Verify readiness via the Ollama HTTP API
3. Inspect available local models
4. Clean up the container when finished